In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import math

# -------------------------------
# 1. Токенізатор (BPE із HuggingFace)
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# -------------------------------
# 2. Датасет (простий приклад)
# -------------------------------
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# -------------------------------
# 3. Синусоїдальне позиційне кодування
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

# -------------------------------
# 4. Модель трансформера (енкодер)
# -------------------------------
class TransformerEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, num_classes=2, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        x = self.positional_encoding(x)

        if attention_mask is not None:
            key_padding_mask = attention_mask == 0  # (batch, seq)
        else:
            key_padding_mask = None

        out = self.encoder(x, src_key_padding_mask=key_padding_mask)
        cls_repr = out[:, 0, :]  # беремо перший токен як "CLS"
        return self.classifier(cls_repr)

# -------------------------------
# 5. Перевірка роботи (для навчання потрібно додати training loop)
# -------------------------------
texts = [
    "Congratulations! You have won a free ticket",
    "Hello, can we meet tomorrow?",
    "Free entry in 2 a weekly competition",
    "See you at the meeting"
]
labels = [1, 0, 1, 0]  # 1 = спам, 0 = не спам

dataset = TextClassificationDataset(texts, labels, tokenizer, max_len=16)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

model = TransformerEncoderClassifier(vocab_size=tokenizer.vocab_size)

for batch in dataloader:
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    labels = batch["label"]
    print("Labels:", labels)
    outputs = model(input_ids, attention_mask)
    print("Logits:", outputs)
    print("Pred:", torch.argmax(outputs, dim=1))
    break

Labels: tensor([1, 0, 1, 0])
Logits: tensor([[-0.4729, -0.0187],
        [-0.0490, -0.0120],
        [-0.3615, -0.2264],
        [-0.2143, -0.2305]], grad_fn=<AddmmBackward0>)
Pred: tensor([1, 1, 1, 0])
